In [ ]:
from __future__ import annotations

import ast
import contextlib
import importlib.util
import os
import sys
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from IPython.display import Markdown, display

try:
    import torch
except Exception:
    torch = None

try:
    import h5py
except Exception:
    h5py = None

try:
    from sklearn.model_selection import train_test_split
except Exception:
    train_test_split = None

pd.set_option('display.max_colwidth', 160)


In [ ]:
CURRENT_DIR = Path(os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / 'Uniform').exists() else CURRENT_DIR.parent
CHECKPOINT_ROOT = REPO_ROOT
H5_DATASET_ROOT = REPO_ROOT / 'Data_gen' / 'output'
SPLIT_SEED = 42
SPLIT_SIZE = 0.2

import sys as _sys
if str(CURRENT_DIR) not in _sys.path:
    _sys.path.insert(0, str(CURRENT_DIR))
import eval_helpers as eh

OUTPUT_DIR = CURRENT_DIR / 'eval_outputs'
FIGURES_DIR = CURRENT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXAMPLE_DIR_CANDIDATES = [
    Path('/home/runner/Desktop/Disc_lifing_paper_v2/Comparison/Examples'),
    Path('/Desktop/Disc_lifing_paper_v2/Comparison/Examples'),
    REPO_ROOT / 'Comparison' / 'Examples',
]

# PointNetMLPJoint_FP only exists for Zonal/Edge today. It is included wherever discovered.
TARGET_FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint', 'PointNetMLPJoint_headfeat', 'PointNetMLPJoint_FP']
MAIN_FAMILIES = ['ArGEnT_self_att_noSDF', 'PointNetMLPJoint', 'PointNetMLPJoint_FP']
MODEL_COLORS = {
    'ArGEnT_self_att_noSDF': '#1f77b4',
    'PointNetMLPJoint': '#ff7f0e',
    'PointNetMLPJoint_headfeat': '#2ca02c',
    'PointNetMLPJoint_FP': '#d62728',
}

if not (REPO_ROOT / 'Uniform').exists() or not (REPO_ROOT / 'Zonal').exists():
    raise RuntimeError(f'Could not locate repository root from {CURRENT_DIR}')